In [ ]:
import re
from pathlib import Path
from collections import defaultdict
from io import BytesIO

import torch
import timm
from PIL import Image
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Config ---
IMAGE_DIR = Path("../data/raw/BarHill2026/Bar Hill 2026")
MODEL_NAME = "hf-hub:BVRA/MegaDescriptor-L-384"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TOP_K = 5

# --- Load MegaDescriptor ---
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0).to(DEVICE).eval()
data_cfg = timm.data.resolve_data_config({}, model=model)
transform = timm.data.create_transform(**data_cfg)

# --- Collect files and labels ---
id_pattern = re.compile(r"(GCN\d+)")
filenames = sorted(f.name for f in IMAGE_DIR.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
labels = [id_pattern.search(f).group(1) for f in filenames]

# --- Extract embeddings ---
embeddings = []
with torch.no_grad():
    for fname in filenames:
        img = Image.open(IMAGE_DIR / fname).convert("RGB")
        x = transform(img).unsqueeze(0).to(DEVICE)
        emb = model(x).squeeze(0).cpu().numpy()
        embeddings.append(emb / np.linalg.norm(emb))  # normalize for cosine sim

embeddings = np.stack(embeddings)  # (n_images, dim)
labels = np.array(labels)
filenames = np.array(filenames)

# --- Cosine similarity between all image pairs ---
sim_matrix = embeddings @ embeddings.T  # already normalized -> cosine similarity

# --- For each query newt, find top-K most similar *other* newts (best single-image match) ---
unique_ids = sorted(set(labels))

def top_matches_for_newt(query_id, k=TOP_K):
    query_idx = np.where(labels == query_id)[0]
    other_idx = np.where(labels != query_id)[0]

    # best similarity per other image, only considering query's images
    sub_sim = sim_matrix[np.ix_(query_idx, other_idx)]  # (n_query_imgs, n_other_imgs)
    best_per_other_img = sub_sim.max(axis=0)  # best match across query's images

    # aggregate to best score per other newt
    scores_by_newt = defaultdict(float)
    best_img_by_newt = {}
    for score, idx in zip(best_per_other_img, other_idx):
        newt_id = labels[idx]
        if score > scores_by_newt[newt_id]:
            scores_by_newt[newt_id] = score
            best_img_by_newt[newt_id] = filenames[idx]

    ranked = sorted(scores_by_newt.items(), key=lambda x: -x[1])[:k]
    return [(newt_id, score, best_img_by_newt[newt_id]) for newt_id, score in ranked]

# --- Interactive dropdown viz ---
dropdown = widgets.Dropdown(options=unique_ids, description="Newt:")
output = widgets.Output()

def on_change(change):
    with output:
        clear_output(wait=True)
        query_id = change["new"]
        query_img = filenames[np.where(labels == query_id)[0][0]]
        print(f"Query: {query_id}  (showing: {query_img})")
        display(Image.open(IMAGE_DIR / query_img).resize((200, 200)))

        print(f"\nTop {TOP_K} most similar newts:")
        matches = top_matches_for_newt(query_id)

        row = []
        for newt_id, score, best_img in matches:
            img = Image.open(IMAGE_DIR / best_img).resize((150, 150))
            buf = BytesIO()
            img.save(buf, format="PNG")
            img_widget = widgets.Image(value=buf.getvalue(), format="png", width=150, height=150)
            label_widget = widgets.Label(f"{newt_id} ({score:.3f})")
            row.append(widgets.VBox([img_widget, label_widget]))

        display(widgets.HBox(row))

In [8]:
dropdown.observe(on_change, names="value")
display(dropdown, output)
on_change({"new": unique_ids[0]})  # trigger initial render

Dropdown(description='Newt:', options=(np.str_('GCN001'), np.str_('GCN002'), np.str_('GCN003'), np.str_('GCN00…

Output()